# B1 · A tree on the Titanic: overfitting, tuning and readable rules

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Build a pipeline with a classification tree on the Titanic data (imputation, encoding of the categoricals with one-hot or ordinal), measure how much a tree **without limits** overfits, choose `max_depth`, `min_samples_leaf` and `ccp_alpha` with `GridSearchCV`
and read the rules of the final tree.

## Dataset

`ds.titanic()` (1309 passengers: `pclass, sex, age, siblings_spouses, parents_children, fare, port, deck, title`; target `survived`), stratified 75/25 split. `age` and `deck` have missing values. Validation: `StratifiedKFold(5, shuffle=True, random_state=0)`, AUC.

## Educational purpose

See in a real case that an unregularized tree is a bad model (it memorizes), that a few parameters turn it into a good readable model, and that the choice of the encoding of the categoricals matters little. Also: a tree is the only model of the course whose rules you can print.

## Task

1. Write `build(encoding, **tree_params)`: `Pipeline` of a `ColumnTransformer` (numeric `pclass, age, siblings_spouses, parents_children, fare` with `SimpleImputer(strategy="median")`; categorical `sex, port, deck, title` with `SimpleImputer(strategy="most_frequent")` and `OneHotEncoder(handle_unknown="ignore")` if `encoding == "ohe"`, or `OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)` if `"ord"`; `sparse_threshold=0`) and `DecisionTreeClassifier(random_state=0, **tree_params)`.
2. Compute `cv_auc` (DataFrame, index: `no limits`, `max_depth=3`, `max_depth=5`, `min_samples_leaf=10`, `min_samples_leaf=20`, `ccp_alpha=0.005`; columns `ohe` and `ord`) with the mean cross-validation AUC.
3. `GridSearchCV` (same `cv`, `scoring="roc_auc"`) on the `ohe` pipeline with `max_depth` in `[2, 3, 4, 5, 6, 8, None]`, `min_samples_leaf` in `[1, 5, 10, 20, 40]` and `ccp_alpha` in `[0, 0.002, 0.005, 0.01]`: save `best` (dictionary of the parameters without the prefix), `test_auc` (test AUC) and `n_leaves`.
4. Read the rules: save `root` (name of the feature at the root, from `get_feature_names_out()` of the preprocessing part) and `rules` (the `export_text` string with the feature names) of the final tree; print the first lines.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]

## Your work

In [ ]:
def build(encoding, **tree_params):
    ...

In [ ]:
cv_auc = pd.DataFrame(index=["no limits", "max_depth=3", "max_depth=5", "min_samples_leaf=10", "min_samples_leaf=20", "ccp_alpha=0.005"], columns=["ohe", "ord"], dtype=float)
cv_auc

In [ ]:
best = None
test_auc = None
n_leaves = None

In [ ]:
root = None
rules = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Names: `names = list(model[:-1].get_feature_names_out())`; `export_text(model[-1], feature_names=names)`; root: `names[model[-1].tree_.feature[0]]`. Parameters after the search: `{k.split("__")[1]: v for k, v in search.best_params_.items()}`.'):
    check(cv_auc.notna().all().all() and cv_auc.loc["no limits"].max() < 0.76, "the tree without limits has AUC below 0.76 (overfitting)", "DecisionTreeClassifier(random_state=0) with no parameters")
    check(cv_auc.loc["max_depth=3"].min() > 0.83 and cv_auc.drop("no limits").min().min() > 0.8, "with any regularization the AUC goes above 0.8 (over 0.83 with max_depth=3)", "compare the rows")
    check(abs(cv_auc.loc["max_depth=3", "ohe"] - cv_auc.loc["max_depth=3", "ord"]) < 0.02, "the encoding (ohe or ordinal) changes little: less than 0.02 with max_depth=3", "compare the ohe and ord columns")
    check(set(best) == {"max_depth", "min_samples_leaf", "ccp_alpha"} and best["max_depth"] in (3, 4, 5) and test_auc > 0.83, "the search picks a depth between 3 and 5; test AUC above 0.83", "keys without the prefix 'decisiontreeclassifier__'")
    check(3 <= n_leaves <= 40, "the final tree is small: from 3 to 40 leaves", "get_n_leaves() of the last step")
    check(isinstance(root, str) and ("sex" in root or "title" in root) and "class:" in rules and rules.count("\n") >= 6, "the root is `sex` or `title`; the rules are a text with the classes", "export_text(tree, feature_names=names)")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_titanic_tree_tuning_and_rules.ipynb`. Open it after you pass the check, or after you have really tried.